# ML-04 — Search Intelligence Data Contract: Refresh / Content Opportunity Scoring

This notebook defines and checks a data contract for ranking content pages for human review. It uses aggregate counts and metrics only; do not print or commit raw rows, client names, URLs, or search queries.

## The five contract answers
1. **One row:** one content item for one client on one report date in the daily performance fact table.
2. **Source table:** `fact_content_daily_performance` only for this first version; no dimension or query-level table is joined.
3. **Time window:** March 2026 is the development feature month and April is its outcome month. May features → June outcomes are a sealed test pair.
4. **Target/proxy:** flag a content item when next-month GSC impressions are at least 20% below feature-month impressions. This is a directional proxy for review prioritization, not a causal claim.
5. **Deliberate exclusion:** all outcome-month metrics are excluded from honest predictors because they would reveal future information; pseudonymous IDs are context/grouping keys only.

**Plan fixed before evaluation:** the development pair is March → April 2026. May → June 2026 is held out as a sealed temporal test; June is not used to invent or tune the label.

The warehouse is gated. Put a Hugging Face **READ** token in Colab Secrets as `HF_TOKEN`; never put the token in a notebook cell or commit it.

In [ ]:
# This notebook is designed for Colab. Do not print or store the token value.
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    print("HF_TOKEN configured in Colab Secrets:", bool(token))
except Exception:
    print("Colab Secrets not available here; the connection cell will prompt for a token.")

In [ ]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

In [ ]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np
from huggingface_hub import HfApi

# Read the token from Colab Secrets or the environment; never print it.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token (input hidden): ")
if not HF_TOKEN or not HF_TOKEN.strip():
    raise ValueError("HF_TOKEN is empty. Add a Hugging Face READ token to Colab Secrets named HF_TOKEN.")

# Check identity and dataset permissions before DuckDB tries to read a parquet file.
try:
    hf_api = HfApi()
    hf_api.whoami(token=HF_TOKEN)
    hf_api.dataset_info("FlyRank/internship-warehouse", token=HF_TOKEN)
except Exception as exc:
    raise RuntimeError(
        "Hugging Face authentication/access check failed. Confirm HF_TOKEN is a valid READ token "
        "and that its account has been granted access to the FlyRank/internship-warehouse dataset. "
        "If access was just granted, create/update the token and rerun this cell. The token value was not printed."
    ) from exc
print("Hugging Face identity and dataset metadata access verified.")

con = duckdb.connect()
# Register hf:// support and the Hugging Face secret type before creating the secret.
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
escaped_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
del escaped_token

REL = "hf://datasets/FlyRank/internship-warehouse"
def month_source(yyyy_mm):
    return f"read_parquet('{REL}/fact_content_daily_performance/month={yyyy_mm}/*.parquet')"

march_src = month_source("2026-03")
try:
    schema = con.sql(f"DESCRIBE SELECT * FROM {march_src}").df()
except Exception as exc:
    raise RuntimeError(
        "DuckDB could not read the March 2026 parquet partition. If the Hugging Face check above passed "
        "but this returns HTTP 403, verify that the same HF account/token has file-download access to the "
        "private dataset, and that any dataset access terms have been accepted. This is an access issue, "
        "not a schema/SQL error."
    ) from exc
available_columns = set(schema["column_name"])
required_columns = {
    "report_date", "client_hash_id", "content_hash_id",
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "gsc_data_available", "ga4_data_available", "ga4_sessions"
}
missing_columns = sorted(required_columns - available_columns)
if missing_columns:
    raise ValueError("Warehouse schema differs. Missing columns: " + ", ".join(missing_columns))
print("Schema check passed; required columns are present. No raw rows are displayed.")

## 1. Unit of analysis + time window

**Contract.** One row in the daily fact table represents one `report_date × client_hash_id × content_hash_id` combination. The development feature window is **1–31 March 2026**, with April 2026 as the next-month outcome. The sealed test uses **May 2026** features and **June 2026** outcomes.

The primary source is `fact_content_daily_performance`. Pseudonymous IDs are for grouping and joining only. The next-month outcome is never a final model feature.

In [ ]:
DEVELOPMENT_FEATURE_MONTH = "2026-03"
DEVELOPMENT_OUTCOME_MONTH = "2026-04"
SEALED_TEST_FEATURE_MONTH = "2026-05"
SEALED_TEST_OUTCOME_MONTH = "2026-06"
assert (DEVELOPMENT_FEATURE_MONTH, DEVELOPMENT_OUTCOME_MONTH) == ("2026-03", "2026-04")
assert (SEALED_TEST_FEATURE_MONTH, SEALED_TEST_OUTCOME_MONTH) == ("2026-05", "2026-06")
print("Contract windows are set. June is reserved for final evaluation.")

## 2. Fields: feature / label / context / excluded

**At most five honest features, all measured by the end of the feature month:**
- `prev_month_impressions` — total GSC impressions during the feature month. **Available when:** that month's daily GSC measurements have landed and `gsc_data_available IS TRUE`.
- `prev_month_clicks` — total GSC clicks during the feature month. **Available when:** that month's daily GSC measurements have landed and GSC availability is true.
- `prev_month_avg_position` — mean positive GSC average position during the feature month. **Available when:** the feature-month position observations have landed; zero/non-positive positions are treated as unavailable.
- `prev_month_ga4_sessions` — total GA4 sessions during the feature month. **Available when:** GA4 measurements are available; rows with `ga4_data_available` false or NULL are not treated as measured zero.
- `prev_month_ga4_days_available` — number of days with GA4 availability in the feature month. **Available when:** the daily availability flags for the feature month have landed; this describes coverage, not engagement itself.

**Label / proxy:** `declined_next_month` is 1 when next-month impressions are less than 80% of feature-month impressions, else 0. It is computed only after the outcome month ends.

**Context only:** `client_hash_id`, `content_hash_id`, `report_date`, month markers, availability flags, and coverage counts used for checks. IDs and dates are never model inputs.

**Excluded from final predictors:** next-month impressions and every other outcome-month metric; pseudonymous IDs as predictive signals; private client names, URLs, and queries; and product-decision scores/flags. The deliberately leaked label-derived feature appears only in the leakage demonstration and is excluded from the final feature list.

In [ ]:
FEATURE_CANDIDATES = [
    "prev_month_impressions", "prev_month_clicks", "prev_month_avg_position",
    "prev_month_ga4_sessions", "prev_month_ga4_days_available",
]
LABEL_COLUMN = "declined_next_month"
# This intentionally label-derived column is created only for the trap demonstration.
LEAKED_COLUMN = "leaked_decline_rule"
assert len(FEATURE_CANDIDATES) <= 5 and LEAKED_COLUMN not in FEATURE_CANDIDATES
print("Honest feature candidates (max 5):", FEATURE_CANDIDATES)
print("Outcome proxy:", LABEL_COLUMN)
print("Leakage-only column excluded from honest features:", LEAKED_COLUMN)

## 3. Verify the contract with exactly three verification queries

The three verification queries below use the March 2026 partition. Query 1 checks the claimed grain; query 2 reports the March row count and date span (plus aggregate missingness); query 3 checks availability with `IS TRUE`. Outputs are aggregate-only. The feature-frame and model cells after these checks are separate analysis steps, not additional contract-verification queries.

### Verification query 1 — test the row grain

In [ ]:
grain_duplicates = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_at_grain
    FROM {march_src}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate grain groups returned:", len(grain_duplicates))
if grain_duplicates.empty:
    print("Observed: no duplicate report_date × client × content groups in March.")
else:
    print("Observed: duplicate groups exist; investigate before relying on this grain.")
    display(grain_duplicates[["rows_at_grain"]])

### Verification query 2 — row counts, date span, and missingness

In [ ]:
counts_and_missing = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content_ids,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS missing_gsc_impressions,
        COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS missing_gsc_clicks,
        COUNT(*) FILTER (WHERE gsc_avg_position IS NULL OR gsc_avg_position = 0) AS missing_or_no_position,
        COUNT(*) FILTER (WHERE ga4_sessions IS NULL) AS missing_ga4_sessions
    FROM {march_src}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()
display(counts_and_missing)

### Verification query 3 — availability flags (use `IS TRUE`)

In [ ]:
availability_counts = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_not_true_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_not_true_rows
    FROM {march_src}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()
display(availability_counts)
print("IS NOT TRUE includes FALSE and NULL; neither is treated as measured availability.")

## 4. Data limits

- **Unbalanced history:** clients start on different dates. March 2026 is a fixed development slice, not proof that every client has equally long history.
- **GSC-only early rows:** GA4 metrics are used only when `ga4_data_available IS TRUE`. A false or NULL flag is not evidence of zero sessions.
- **Window overlap:** next-month impressions define the proxy label. Any metric from that outcome month would leak the answer if used as a feature. The query-level 90-day table also overlaps recent months, so it is not joined here.
- **Selection and measurement limits:** pages without usable GSC measurements cannot support this label in the same way. The proxy measures observed impression decline, not cause, content quality, or Google's ranking algorithm.
- **Decision-support only:** the analysis can describe measured patterns in this data slice; it cannot establish that a content change caused an outcome.

## Feature frame + deliberate label-leakage trap

The feature frame uses four monthly partitions to create March → April development examples and May → June sealed-test examples. Each row in this modelling frame is one client-content pair for a feature month, not one daily fact row. The target is a 20% drop in next-month impressions, with a minimum of 100 feature-month impressions and measured GSC coverage in both months.

The trap deliberately creates one feature from the same future-window rule that defines the label. It is a label-derived shortcut and is **not** a valid predictor. We score it only to show how a contaminated feature can make a model look nearly perfect; the final feature list and reported honest score exclude it.

In [ ]:
month_union = " UNION ALL ".join(
    f"SELECT * FROM {month_source(m)}"
    for m in ["2026-03", "2026-04", "2026-05", "2026-06"]
)
feature_frame_sql = f"""
WITH daily AS ({month_union}),
monthly AS (
    SELECT
        CAST(date_trunc('month', report_date) AS DATE) AS month_start,
        client_hash_id, content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions END) AS impressions,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks END) AS clicks,
        AVG(CASE WHEN gsc_data_available IS TRUE AND gsc_avg_position > 0
                 THEN gsc_avg_position END) AS avg_position,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions END) AS ga4_sessions,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_days_available,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_available
    FROM daily
    GROUP BY 1, 2, 3
),
paired AS (
    SELECT
        p.client_hash_id, p.content_hash_id, p.month_start AS feature_month,
        p.impressions AS prev_month_impressions,
        p.clicks AS prev_month_clicks,
        p.avg_position AS prev_month_avg_position,
        p.ga4_sessions AS prev_month_ga4_sessions,
        p.ga4_days_available AS prev_month_ga4_days_available,
        p.gsc_days_available AS prev_month_gsc_days_available,
        n.impressions AS next_month_impressions,
        n.gsc_days_available AS next_month_gsc_days_available
    FROM monthly p
    JOIN monthly n
      ON n.client_hash_id = p.client_hash_id
     AND n.content_hash_id = p.content_hash_id
     AND n.month_start = p.month_start + INTERVAL '1 month'
    WHERE (p.month_start = DATE '2026-03-01' AND n.month_start = DATE '2026-04-01')
       OR (p.month_start = DATE '2026-05-01' AND n.month_start = DATE '2026-06-01')
)
SELECT *,
       CAST(next_month_impressions < 0.8 * prev_month_impressions AS INTEGER) AS declined_next_month
FROM paired
WHERE prev_month_impressions >= 100
  AND next_month_impressions IS NOT NULL
  AND prev_month_gsc_days_available > 0
  AND next_month_gsc_days_available > 0
"""
feature_frame = con.sql(feature_frame_sql).df()
feature_frame["feature_month"] = pd.to_datetime(feature_frame["feature_month"])

# Deliberate leakage: re-create the target rule as a feature using the future outcome.
# This column is only for the experiment below; never include it in the honest model.
feature_frame[LEAKED_COLUMN] = (
    feature_frame["next_month_impressions"] < 0.8 * feature_frame["prev_month_impressions"]
).astype("int8")

dev = feature_frame[feature_frame["feature_month"].dt.strftime("%Y-%m") == "2026-03"].copy()
test = feature_frame[feature_frame["feature_month"].dt.strftime("%Y-%m") == "2026-05"].copy()
print(f"Development rows (March features → April outcome): {len(dev):,}")
print(f"Sealed test rows (May features → June outcome): {len(test):,}")
print("No content rows or identifiers are displayed.")
print("Observed development label rate:", round(float(dev[LABEL_COLUMN].mean()), 4) if len(dev) else "no eligible rows")
print("Observed sealed-test label rate:", round(float(test[LABEL_COLUMN].mean()), 4) if len(test) else "no eligible rows")

# Aggregate-only feature-frame summary: no page-level records or IDs are displayed.
summary_columns = FEATURE_CANDIDATES + [LABEL_COLUMN]
summary = (
    feature_frame.groupby(feature_frame["feature_month"].dt.strftime("%Y-%m"))[summary_columns]
    .agg(["count", "median"])
    .round(2)
)
print("Feature-frame counts and medians by feature month (aggregate only):")
display(summary)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score

honest_features = [c for c in FEATURE_CANDIDATES if c in dev.columns and dev[c].notna().any()]
assert LEAKED_COLUMN not in honest_features
if not honest_features:
    raise ValueError("No usable honest features were found in the development slice.")
if len(dev) == 0 or len(test) == 0:
    raise ValueError("No eligible development/test rows. Check warehouse access, schema, and month partitions.")
if dev[LABEL_COLUMN].nunique() < 2:
    raise ValueError("Development slice contains one label class; do not report a model score.")
if test[LABEL_COLUMN].nunique() < 2:
    raise ValueError("Sealed test contains one label class; balanced accuracy and precision/recall would be misleading for this evaluation.")

def fit_and_score(feature_columns, experiment):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=150, random_state=42, class_weight="balanced", n_jobs=-1)
    )
    model.fit(dev[feature_columns], dev[LABEL_COLUMN])
    pred = model.predict(test[feature_columns])
    return {
        "experiment": experiment,
        "features_used": len(feature_columns),
        "balanced_accuracy": round(float(balanced_accuracy_score(test[LABEL_COLUMN], pred)), 4),
        "precision": round(float(precision_score(test[LABEL_COLUMN], pred, zero_division=0)), 4),
        "recall": round(float(recall_score(test[LABEL_COLUMN], pred, zero_division=0)), 4),
    }

honest_result = fit_and_score(honest_features, "HONEST: prior-month features only")
print("Honest feature frame (columns only):", honest_features)
display(pd.DataFrame([honest_result]))
print("Keep this honest score for interpretation; do not compare it as if the leak were a real model.")

In [ ]:
# Deliberate leakage experiment: this feature is computed from next-month impressions
# using the exact rule that defines declined_next_month. It must never be used for prediction.
leaky_features = honest_features + [LEAKED_COLUMN]
leaky_result = fit_and_score(leaky_features, "INVALID LEAKAGE DEMO: label-derived feature included")
display(pd.DataFrame([honest_result, leaky_result]))

FINAL_FEATURES = honest_features.copy()
assert LEAKED_COLUMN not in FINAL_FEATURES
assert "next_month_impressions" not in FINAL_FEATURES
print("Retained honest test score (balanced accuracy):", honest_result["balanced_accuracy"])
print("Leakage-only score is invalid and is shown only to expose the trap.")
print("Final model inputs:", FINAL_FEATURES)

## 5. Self-check

- [ ] The five plain-words contract answers are stated above.
- [ ] Exactly three verification queries ran and their aggregate outputs are visible: grain, count/date span, availability using `IS TRUE`.
- [ ] The feature frame has no more than five honest features, each with an explicit “Available when” statement.
- [ ] The deliberate label-derived feature makes the invalid score look unusually strong; it is absent from `FINAL_FEATURES`, and the honest score is retained.
- [ ] The notebook runs top-to-bottom in Colab using my own approved Hugging Face READ token.
- [ ] The executed notebook is saved and committed here. No tokens, raw rows, datasets, URLs, client names, or private queries are committed.

**Named limitation — unbalanced panel / measurement coverage:** clients and content items do not all have the same history or analytics coverage. This slice is therefore a measured decision-support signal for pages with enough observed GSC data, not a representative claim about every page or a causal estimate.

**Interpretation rule:** use *observed*, *measured*, *directional*, and *decision-support*. Do not claim to predict Google's algorithm or establish causation.